In [5]:
!pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]


# Conexión al laboratorio SQL

In [1]:
from sqlalchemy import create_engine
import sqlalchemy
import mysql.connector as mysql
import pandas as pd
from sqlalchemy import text
import os, getpass

# Setting up the connection

In [ ]:
# Parámetros de conexión
user = os.getenv("DB_USER") or input("Usuario MySQL: ")
password = os.getenv("DB_PASSWORD") or getpass.getpass("Contraseña MySQL: ")
sqlServer = os.getenv("DB_HOST", "localhost")

# Nueva base de datos
db = "VIP_VACACIONES"

# Conexión inicial (sin DB)
connStr = f"mysql+mysqlconnector://{user}:{password}@{sqlServer}"
conn = create_engine(connStr).connect()

# Crear base de datos nueva
conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {db}"))
print("Base de datos creada correctamente")

# Conectamos con el servidor
connStr = f"mysql+mysqlconnector://{user}:{password}@{sqlServer}/{db}"
conn = create_engine(connStr).connect()

print("Conectada dentro de VIP_VACACIONES")


# Crear tablas

In [3]:

conn.execute(text("""
CREATE TABLE ALOJAMIENTO (
  CODIGO_ALOJAMIENTO VARCHAR(10) PRIMARY KEY,
  NOMBRE VARCHAR(100),
  CANT_BANYOS INT,
  CANT_HAB INT,
  SALON VARCHAR(2),
  TERRAZA VARCHAR(2),
  AIREC_ACOND VARCHAR(2),
  PISCINA VARCHAR(2),
  SUPERFICIE INT
);
"""))

conn.execute(text("""
CREATE TABLE UBICACION (
  CODIGO_ALOJAMIENTO VARCHAR(10) PRIMARY KEY,
  PAIS VARCHAR(50),
  CIUDAD VARCHAR(50),
  DIST_METRO DECIMAL(6,2),
  DIST_CENTRO DECIMAL(6,2)
);
"""))

conn.execute(text("""
CREATE TABLE PRECIO (
  CODIGO_ALOJAMIENTO VARCHAR(10) PRIMARY KEY,
  PRECIO_ALQUILER INT,
  RESERVA VARCHAR(2),
  PORCENTAJE_RESERVA INT
);
"""))

conn.execute(text("""
CREATE TABLE PUNTUACION (
  CODIGO_ALOJAMIENTO VARCHAR(10) PRIMARY KEY,
  PUNTOS DECIMAL(4,2),
  AGENCIA VARCHAR(50),
  PUNTOS_AGENCIA DECIMAL(4,2)
);
"""))


ProgrammingError: (mysql.connector.errors.ProgrammingError) 1050 (42S01): Table 'alojamiento' already exists
[SQL: 
CREATE TABLE ALOJAMIENTO (
  CODIGO_ALOJAMIENTO VARCHAR(10) PRIMARY KEY,
  NOMBRE VARCHAR(100),
  CANT_BANYOS INT,
  CANT_HAB INT,
  SALON VARCHAR(2),
  TERRAZA VARCHAR(2),
  AIREC_ACOND VARCHAR(2),
  PISCINA VARCHAR(2),
  SUPERFICIE INT
);
]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [3]:
df_aloj = pd.read_excel("ALOJAMIENTO.xlsx")
df_ubi  = pd.read_excel("UBICACION.xlsx")
df_precio  = pd.read_excel("PRECIO.xlsx")
df_puntuacion  = pd.read_excel("PUNTUACION.xlsx")

df_aloj.to_sql("ALOJAMIENTO", con=conn, if_exists="append", index=False)
df_ubi.to_sql("UBICACION", con=conn, if_exists="append", index=False)
df_precio.to_sql("PRECIO", con=conn, if_exists="append", index=False)
df_puntuacion.to_sql("PUNTUACION", con=conn, if_exists="append", index=False)


20

In [4]:
pd.read_sql("SELECT COUNT(*) FROM ALOJAMIENTO", conn)

,COUNT(*)
0,20


# Primera parte

## PASO 1

### Query 1 - El cliente exige terraza y aire acondicionado, pero no quiere piscina.

In [5]:
pd.read_sql("""
SELECT *
FROM ALOJAMIENTO
WHERE TERRAZA = 'SI'
  AND AIREC_ACOND = 'SI'
  AND PISCINA = 'NO';
""", conn)

,CODIGO_ALOJAMIENTO,NOMBRE,CANT_BANYOS,CANT_HAB,SALON,TERRAZA,AIREC_ACOND,PISCINA,SUPERFICIE
0,ALOJ004,Piso Centro Histórico,2,3,SI,SI,SI,NO,90
1,ALOJ010,Ático del Sol,3,3,SI,SI,SI,NO,88
2,ALOJ015,Dúplex Las Rocas,3,4,SI,SI,SI,NO,120
3,ALOJ018,Piso Parque Central,2,4,SI,SI,SI,NO,100


### Query 2 - No se alojará en estudios que no tengan salón ni en propiedades con solo un baño

In [6]:
pd.read_sql("""
SELECT *
FROM ALOJAMIENTO
WHERE SALON = 'SI'
  AND CANT_BANYOS > 1;
""", conn)

,CODIGO_ALOJAMIENTO,NOMBRE,CANT_BANYOS,CANT_HAB,SALON,TERRAZA,AIREC_ACOND,PISCINA,SUPERFICIE
0,ALOJ001,Villa Sol del Mar,2,3,SI,SI,SI,SI,120
1,ALOJ004,Piso Centro Histórico,2,3,SI,SI,SI,NO,90
2,ALOJ006,Chalet Vista al Lago,3,4,SI,SI,SI,SI,150
3,ALOJ007,Dúplex Jardines del Sur,2,3,SI,SI,SI,SI,130
4,ALOJ009,Casa Rústica Montaña,2,2,SI,SI,NO,NO,85
5,ALOJ010,Ático del Sol,3,3,SI,SI,SI,NO,88
6,ALOJ012,Villa Palmeras,2,3,SI,SI,SI,SI,110
7,ALOJ014,Casa del Mar,3,4,SI,SI,SI,SI,160
8,ALOJ015,Dúplex Las Rocas,3,4,SI,SI,SI,NO,120
9,ALOJ017,Chalet Jardín Secreto,3,5,SI,SI,SI,SI,180


### Query 3 - La propiedad debe tener como mínimo 80m2

In [7]:
pd.read_sql("""
SELECT *
FROM ALOJAMIENTO
WHERE SUPERFICIE >= 80;
""", conn)

,CODIGO_ALOJAMIENTO,NOMBRE,CANT_BANYOS,CANT_HAB,SALON,TERRAZA,AIREC_ACOND,PISCINA,SUPERFICIE
0,ALOJ001,Villa Sol del Mar,2,3,SI,SI,SI,SI,120
1,ALOJ004,Piso Centro Histórico,2,3,SI,SI,SI,NO,90
2,ALOJ006,Chalet Vista al Lago,3,4,SI,SI,SI,SI,150
3,ALOJ007,Dúplex Jardines del Sur,2,3,SI,SI,SI,SI,130
4,ALOJ009,Casa Rústica Montaña,2,2,SI,SI,NO,NO,85
5,ALOJ010,Ático del Sol,3,3,SI,SI,SI,NO,88
6,ALOJ012,Villa Palmeras,2,3,SI,SI,SI,SI,110
7,ALOJ013,Apartamento Moderno,1,2,SI,NO,SI,NO,80
8,ALOJ014,Casa del Mar,3,4,SI,SI,SI,SI,160
9,ALOJ015,Dúplex Las Rocas,3,4,SI,SI,SI,NO,120


### Query 4 - no quiere alojarse en propiedades a menos de 1 km de una estación de metro ni a menos de 2 km del centro.

In [8]:
pd.read_sql("""
SELECT *
FROM UBICACION
WHERE DIST_METRO >= 1
  AND DIST_CENTRO >= 2;
""", conn)

,CODIGO_ALOJAMIENTO,PAIS,CIUDAD,DIST_METRO,DIST_CENTRO
0,ALOJ001,Portugal,Liboa,1.0,2.0
1,ALOJ004,Francia,Marsella,3.0,4.0
2,ALOJ005,Portugal,Oporto,4.0,3.0
3,ALOJ006,Francia,Paris,2.0,2.0
4,ALOJ007,Francia,Toulouse,1.0,3.0
5,ALOJ008,España,Barcelona,2.0,3.0
6,ALOJ009,España,Madrid,2.0,4.0
7,ALOJ010,España,Bilbao,2.0,5.0
8,ALOJ015,España,Málaga,2.0,4.0
9,ALOJ016,Italia,Roma,2.0,2.0


### Query 5 - El rango permitido por el cliente para pagar es entre 1.500€ y 2.000€ la noche.

In [9]:
pd.read_sql("""
SELECT *
FROM PRECIO
WHERE PRECIO_ALQUILER BETWEEN 1500 AND 2000;
""", conn)

,CODIGO_ALOJAMIENTO,PRECIO_ALQUILER,RESERVA,PORCENTAJE_RESERVA
0,ALOJ001,1800,SI,100
1,ALOJ004,1600,SI,25
2,ALOJ008,1500,SI,25
3,ALOJ009,1600,SI,100
4,ALOJ010,1750,SI,25
5,ALOJ011,1825,NO,0
6,ALOJ012,1750,NO,0
7,ALOJ013,1652,NO,0
8,ALOJ014,1941,NO,0
9,ALOJ015,1800,SI,25


### Query 6 - el cliente pide que el % de reserva no sea muy alto, pero tampoco tan bajo (porque le genera inseguridad) por lo que propone un 25%

In [10]:
pd.read_sql("""
SELECT *
FROM PRECIO
WHERE PORCENTAJE_RESERVA = 25;
""", conn)

,CODIGO_ALOJAMIENTO,PRECIO_ALQUILER,RESERVA,PORCENTAJE_RESERVA
0,ALOJ002,780,SI,25
1,ALOJ004,1600,SI,25
2,ALOJ008,1500,SI,25
3,ALOJ010,1750,SI,25
4,ALOJ015,1800,SI,25
5,ALOJ018,1900,SI,25


### Query 7 - Quiere que el alojamiento tenga más de 4,5 puntos de confianza y que la agencia que lo lleva tenga más de 4 puntos

In [11]:
pd.read_sql("""
SELECT *
FROM PUNTUACION
WHERE PUNTOS > 4.5
  AND PUNTOS_AGENCIA > 4;
""", conn)

,CODIGO_ALOJAMIENTO,PUNTOS,AGENCIA,PUNTOS_AGENCIA
0,ALOJ001,5.0,GlobalHome,4.4
1,ALOJ004,4.6,CaribeRent,4.6
2,ALOJ007,4.9,SevillaRooms,4.5
3,ALOJ008,4.8,ChileVive,4.3
4,ALOJ009,4.7,MexicoRent,4.8
5,ALOJ010,4.8,UYAlquila,4.9
6,ALOJ014,4.7,BogotaLodge,4.5
7,ALOJ015,4.9,EasyStay Valencia,4.7
8,ALOJ017,4.9,CuscoRenta,4.4
9,ALOJ018,4.7,MendozaRooms,4.7


## PASO 2

In [12]:
pd.read_sql("""
SELECT *
FROM ALOJAMIENTO
INNER JOIN UBICACION
    ON ALOJAMIENTO.CODIGO_ALOJAMIENTO = UBICACION.CODIGO_ALOJAMIENTO
INNER JOIN PRECIO
    ON ALOJAMIENTO.CODIGO_ALOJAMIENTO = PRECIO.CODIGO_ALOJAMIENTO
INNER JOIN PUNTUACION
    ON ALOJAMIENTO.CODIGO_ALOJAMIENTO = PUNTUACION.CODIGO_ALOJAMIENTO
WHERE TERRAZA = 'SI'
  AND AIREC_ACOND = 'SI'
  AND PISCINA = 'NO'
  AND SALON = 'SI'
  AND CANT_BANYOS > 1
  AND SUPERFICIE >= 80
  AND DIST_METRO >= 1
  AND DIST_CENTRO >= 2
  AND PRECIO_ALQUILER BETWEEN 1500 AND 2000
  AND PORCENTAJE_RESERVA BETWEEN 20 AND 30
  AND PUNTOS > 4.5
  AND PUNTOS_AGENCIA > 4;
""", conn)

,CODIGO_ALOJAMIENTO,NOMBRE,CANT_BANYOS,CANT_HAB,SALON,TERRAZA,AIREC_ACOND,PISCINA,SUPERFICIE,CODIGO_ALOJAMIENTO,...,DIST_METRO,DIST_CENTRO,CODIGO_ALOJAMIENTO,PRECIO_ALQUILER,RESERVA,PORCENTAJE_RESERVA,CODIGO_ALOJAMIENTO,PUNTOS,AGENCIA,PUNTOS_AGENCIA
0,ALOJ004,Piso Centro Histórico,2,3,SI,SI,SI,NO,90,ALOJ004,...,3.0,4.0,ALOJ004,1600,SI,25,ALOJ004,4.6,CaribeRent,4.6
1,ALOJ010,Ático del Sol,3,3,SI,SI,SI,NO,88,ALOJ010,...,2.0,5.0,ALOJ010,1750,SI,25,ALOJ010,4.8,UYAlquila,4.9
2,ALOJ015,Dúplex Las Rocas,3,4,SI,SI,SI,NO,120,ALOJ015,...,2.0,4.0,ALOJ015,1800,SI,25,ALOJ015,4.9,EasyStay Valencia,4.7
3,ALOJ018,Piso Parque Central,2,4,SI,SI,SI,NO,100,ALOJ018,...,2.0,4.0,ALOJ018,1900,SI,25,ALOJ018,4.7,MendozaRooms,4.7


## Tercer paso

Tras aplicar todos los filtros del cliente VIP, he obtenido cuatro propiedades finales (ALOJ004, ALOJ010, ALOJ015 y ALOJ018).

La opción más recomendable es ALOJ015 (Dúplex Las Rocas), ya que además de cumplir todos los requisitos exigidos (terraza, aire acondicionado, sin piscina, mínimo de superficie, buena ubicación, precio adecuado y alta puntuación), destaca por ser la propiedad más amplia, con 120 m2, y por ofrecer una distribución óptima para un grupo familiar grande, con 4 habitaciones y 3 baños.

Por tanto, considero que es la alternativa más completa y cómoda para garantizar una buena estancia para el cliente y su familia.

# Segunda parte

### ¿Cuál es la suma total de metros cuadrados de los alojamientos que tienen piscina?

In [13]:
pd.read_sql("""
SELECT SUM(SUPERFICIE) AS TOTAL_M2_PISCINA
FROM ALOJAMIENTO
WHERE PISCINA = 'SI';
""", conn)

,TOTAL_M2_PISCINA
0,850.0


### ¿Cuál es el alojamiento con mayor superficie?

In [14]:
pd.read_sql("""
SELECT *
FROM ALOJAMIENTO
ORDER BY SUPERFICIE DESC
LIMIT 1;
""", conn)

,CODIGO_ALOJAMIENTO,NOMBRE,CANT_BANYOS,CANT_HAB,SALON,TERRAZA,AIREC_ACOND,PISCINA,SUPERFICIE
0,ALOJ017,Chalet Jardín Secreto,3,5,SI,SI,SI,SI,180


### ¿Cuál es el alojamiento con menor superficie?

In [15]:
pd.read_sql("""
SELECT *
FROM ALOJAMIENTO
ORDER BY SUPERFICIE ASC
LIMIT 1;
""", conn)

,CODIGO_ALOJAMIENTO,NOMBRE,CANT_BANYOS,CANT_HAB,SALON,TERRAZA,AIREC_ACOND,PISCINA,SUPERFICIE
0,ALOJ016,Estudio Urbano,1,1,NO,NO,SI,NO,35


### ¿Cuántos alojamientos tienen salón?

In [16]:
pd.read_sql("""
SELECT COUNT(*) AS TOTAL_CON_SALON
FROM ALOJAMIENTO
WHERE SALON = 'SI';
""", conn)

,TOTAL_CON_SALON
0,16


### ¿Cuántos alojamientos tienen terraza?

In [17]:
pd.read_sql("""
SELECT COUNT(*) AS TOTAL_CON_TERRAZA
FROM ALOJAMIENTO
WHERE TERRAZA = 'SI';
""", conn)

,TOTAL_CON_TERRAZA
0,14


### ¿Cuántos alojamientos hay por cada país según nuestra tabla de UBICACIÓN?

In [18]:
pd.read_sql("""
SELECT PAIS, COUNT(*) AS TOTAL_ALOJAMIENTOS
FROM UBICACION
GROUP BY PAIS;
""", conn)

,PAIS,TOTAL_ALOJAMIENTOS
0,Portugal,4
1,España,8
2,Francia,4
3,Italia,4


### ¿Cuál es el promedio de precios de alquiler de todos los alojamientos según la tabla PRECIO?

In [19]:
pd.read_sql("""
SELECT AVG(PRECIO_ALQUILER) AS PRECIO_MEDIO
FROM PRECIO;
""", conn)

,PRECIO_MEDIO
0,1568.4


### ¿Cuáles son las 3 agencias que tienen la mayor puntuación de agencia (campo a usar: PUNTOS_AGENCIA) según la tabla PUNTUACION?

In [20]:
pd.read_sql("""
SELECT AGENCIA, PUNTOS_AGENCIA
FROM PUNTUACION
ORDER BY PUNTOS_AGENCIA DESC
LIMIT 3;
""", conn)

,AGENCIA,PUNTOS_AGENCIA
0,VivaStay,4.9
1,UYAlquila,4.9
2,TerraHouse,4.9


### Muestra los alojamientos (código y nombre) que: Contengan la palabra ‘Piso’ en su nombre o que tengan piscina y aire acondicionado, pero no pasen los 150 metros de superficie o que tengan más de un baño

In [21]:
pd.read_sql("""
SELECT CODIGO_ALOJAMIENTO, NOMBRE
FROM ALOJAMIENTO
WHERE NOMBRE LIKE '%Piso%'
   OR (PISCINA = 'SI' AND AIREC_ACOND = 'SI' AND SUPERFICIE <= 150)
   OR CANT_BANYOS > 1;
""", conn)

,CODIGO_ALOJAMIENTO,NOMBRE
0,ALOJ001,Villa Sol del Mar
1,ALOJ004,Piso Centro Histórico
2,ALOJ006,Chalet Vista al Lago
3,ALOJ007,Dúplex Jardines del Sur
4,ALOJ009,Casa Rústica Montaña
5,ALOJ010,Ático del Sol
6,ALOJ012,Villa Palmeras
7,ALOJ014,Casa del Mar
8,ALOJ015,Dúplex Las Rocas
9,ALOJ017,Chalet Jardín Secreto


### Muestra una consulta que muestre el código de alojamiento, el nombre, el precio y una columna donde veamos el precio del alquiler con el símbolo del euro al final de cada importe, esta nueva columna se debe llama PRECIO_MODIF

In [22]:
pd.read_sql("""
SELECT 
    CODIGO_ALOJAMIENTO,
    PRECIO_ALQUILER,
    CONCAT(PRECIO_ALQUILER, ' €') AS PRECIO_MODIF
FROM PRECIO;
""", conn)

,CODIGO_ALOJAMIENTO,PRECIO_ALQUILER,PRECIO_MODIF
0,ALOJ001,1800,1800 €
1,ALOJ002,780,780 €
2,ALOJ003,800,800 €
3,ALOJ004,1600,1600 €
4,ALOJ005,900,900 €
5,ALOJ006,1200,1200 €
6,ALOJ007,1300,1300 €
7,ALOJ008,1500,1500 €
8,ALOJ009,1600,1600 €
9,ALOJ010,1750,1750 €


### Muestra una columna sobre la tabla PRECIO que se llame CATEGORIA_PRECIO y que indique la palabra ‘Bajo’ si el precio está entre 780 y 1000, ‘Medio’ si el precio está entre 1000 y 1700 y ‘Alto’ si es mayor a 1700

In [23]:
pd.read_sql("""
SELECT 
    CODIGO_ALOJAMIENTO,
    PRECIO_ALQUILER,
    CASE
        WHEN PRECIO_ALQUILER BETWEEN 780 AND 1000 THEN 'Bajo'
        WHEN PRECIO_ALQUILER BETWEEN 1000 AND 1700 THEN 'Medio'
        WHEN PRECIO_ALQUILER > 1700 THEN 'Alto'
    END AS CATEGORIA_PRECIO
FROM PRECIO;
""", conn)

,CODIGO_ALOJAMIENTO,PRECIO_ALQUILER,CATEGORIA_PRECIO
0,ALOJ001,1800,Alto
1,ALOJ002,780,Bajo
2,ALOJ003,800,Bajo
3,ALOJ004,1600,Medio
4,ALOJ005,900,Bajo
5,ALOJ006,1200,Medio
6,ALOJ007,1300,Medio
7,ALOJ008,1500,Medio
8,ALOJ009,1600,Medio
9,ALOJ010,1750,Alto
